# Example to run $\mathsf{VaSST}$

### Import from $\mathsf{VaSST}$

In [1]:
# import everything from VaSST
from VaSST import *

### Simulating data 

- Covariate generation: $x_{i, j} \sim \mathrm{Unif}(-\pi, \pi)$ for all $i, j$.

- Response generation: $\mathbf{y} = 6\sin(\mathbf{x}_0)\cos(\mathbf{x}_1) + \pmb\epsilon$, where $\pmb\epsilon \sim \mathrm{N}(\pmb 0_n, \sigma^{2}\mathbf{I}_n)$ with $\sigma^{2} = 0.1^{2}$, $n=2000$, and $p=2$.

In [2]:
# ----------------------------
# 1) simulate dataset
# ----------------------------
def simulate_data(n=2000, noise_std=0.1, seed=0, device="cpu", dtype=torch.float32):
    g = torch.Generator(device=device)
    g.manual_seed(seed)

    # features and response generation
    X = (2.0 * torch.pi) * torch.rand(n, 2, generator=g, device=device, dtype=dtype) - torch.pi
    y_clean = 6.0 * torch.sin(X[:, 0]) * torch.cos(X[:, 1])
    y = y_clean + noise_std * torch.randn(n, generator=g, device=device, dtype=dtype)

    return X, y, y_clean

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float32

X, y, y_clean = simulate_data(
    n=2000,
    noise_std=0.1,
    seed=0,
    device=device,
    dtype=dtype,
)

### Build the $\mathsf{VaSST}$ model

In [3]:
# ----------------------------
# 2) build VaSST model
# ----------------------------
# custom operator set.
ops = make_operator_set(["mul", "sin", "cos"])

model = VaSST(
    n_features=2,
    n_trees=3,        
    depth=3,         
    operators=ops,
    alpha_split=0.95,
    delta0=2.0,
    eta_op_prior=1.0,
    eta_ft_prior=1.0,
    blr_a0=2.0,
    blr_b0=2.0,
    blr_sigma0_scale=1.0,
    tau_e=1.0,
    tau_op=1.0,
    tau_ft=1.0,
    value_clip=1e3,
    use_tanh_clip=True,
    logits_clip=10.0,
    device=torch.device(device),
    dtype=dtype,
).to(device)

### Train the $\mathsf{VaSST}$ model

In [4]:
# ----------------------------
# 3) train
# ----------------------------
cfg = TrainConfig(
    lr=5e-5,
    n_steps=1000,
    mc_samples=8,
    grad_clip=1.0,
    tau_start=1.0,
    tau_end=0.5,
    tau_anneal_steps=600,
    jitter=1e-3,
    log_every=100,
    kl_warmup_steps=300,
    kl_start=0.0,
    kl_end=1.0,
)

logs = train_VaSST(
    model=model,
    X=X,
    y=y,
    cfg=cfg,
    include_intercept=True,
    verbose=False,
    print_every=100,
    use_progress_bar=False,
)

### Rank sampled hard trees according to LMPSE and print top $\mathsf{r}=10$ ranked results

In [5]:
# ----------------------------
# 4) rank hard samples by LMPSE
# ----------------------------
expr_top_lmpse, beta_top_lmpse, expr_top_logm, beta_top_logm = rank_hard_tree_samples_by_lmpse(
    model=model,
    X=X,
    y=y,
    n_samples=5000,
    include_intercept=True,
    jitter=1e-3,
    standardize_xy=False,
    top_k=10,
    feature_names=["x0", "x1"],
)


=== TOP samples by LMPSE ===
 rank  sample_id     rmse  sigma2_mean  log_marginal  log_tree_prior      lmpse                        tree_0              tree_1              tree_2
    1       1844 0.101058     0.030139    657.271484      -21.818029 635.453430                            x0 (cos(x1) * sin(x0))                  x0
    2       1266 0.101055     0.030130    654.455566      -21.767271 632.688293           (sin(x0) * cos(x1))                  x0             cos(x0)
    3       4092 0.101023     0.030127    653.607910      -21.818029 631.789856           (sin(x0) * cos(x1))                  x0                  x1
    4       3205 0.100986     0.030119    652.761719      -25.103971 627.657776           (sin(x0) * cos(x1))           (x1 * x1)           (x0 * x1)
    5       4713 0.101056     0.030131    654.480469      -31.452791 623.027649           (cos(x1) * sin(x0))      sin((x0 * x1))                  x0
    6       1881 0.101037     0.030128    655.967773      -36.067696 6